# 04 · Segmenter : qui sont nos clients ?

**Séance 6 · deux blocs d'environ 25 minutes en séance, 30 minutes en autonomie.**

Première moitié de la question directrice : **« qui sont nos clients ? »** Jusqu'ici nous avions une cible (`Response`) et un modèle qui apprenait à la prédire. Ici, **pas de cible** : on cherche à regrouper les clients qui se ressemblent, pour parler à chaque groupe différemment. C'est l'apprentissage **non supervisé**, et l'algorithme s'appelle **k-means**.

Trois avertissements, qui sont le fil du notebook :

- un segment est une **construction** : il dépend des variables choisies, de leur échelle et du nombre k de groupes demandé ;
- les segments obtenus ici sont des **découpages stables et utiles, pas des groupes naturels** : les indicateurs le diront ;
- le **nom** d'un segment est une interprétation, à justifier par ses chiffres.

**Bloc 1 — Construire des segments** : la distance entre clients, le poids des variables, le formulaire, avant / après standardisation.
**Bloc 2 — Interpréter et valider** : combien de groupes, quels profils, tiennent-ils sur ce qu'on n'a pas utilisé, quels noms.

## Bloc 1 — Construire des segments

### 1. Charger et préparer

Même première cellule, même préparation. Pas de séparation apprentissage / test ici : on ne prédit rien, on décrit **tous** les clients préparés.

In [ ]:
# Première cellule de chaque notebook : récupérer le module du cours, puis charger le fichier brut.
import os
import urllib.request

URL_MODULE = "https://raw.githubusercontent.com/2501Pr0ject/Data4Sup2Pub/main/notebooks/preparation.py"
if not os.path.exists("preparation.py"):
    urllib.request.urlretrieve(URL_MODULE, "preparation.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import preparation as prep

brut = prep.charger()
df = prep.preparer(brut, verbose=False)
print("Clients à segmenter :", len(df))

### 2. Qu'est-ce que segmenter ?

**Question marketing :** *comment regrouper des clients « qui se ressemblent » ?*

**La notion.** Deux clients se ressemblent si leurs valeurs sont proches sur les variables qu'on a choisies : on mesure une **distance** entre eux. K-means répète trois gestes : placer k centres, affecter chaque client au centre le plus proche, déplacer chaque centre au milieu de ses clients ; jusqu'à ce que plus rien ne bouge. Le résultat dépend des **variables**, de leur **échelle** et de **k**.

Variables de départ (configuration par défaut) : `Depenses_total` (ce que le client achète), `NumDealsPurchases` (sa sensibilité aux promotions), `Enfants` (son foyer), `Age`. Deux fonctions serviront tout le notebook : `segmenter()` et `profil()`.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

VARIABLES_DEFAUT = ["Depenses_total", "NumDealsPurchases", "Enfants", "Age"]

def segmenter(variables, k, standardiser=True, graine=prep.GRAINE):
    # Retourne le numéro de segment de chaque client, la silhouette et l'inertie
    X = df[variables].to_numpy(dtype=float)
    if standardiser:
        X = StandardScaler().fit_transform(X)      # chaque variable : moyenne 0, écart-type 1
    modele = KMeans(n_clusters=k, n_init=10, random_state=graine).fit(X)
    return modele.labels_, silhouette_score(X, modele.labels_), modele.inertia_

def profil(segments, variables):
    # Effectif, part et moyennes par segment, plus deux variables NON utilisées (validation externe)
    g = df.assign(segment=segments).groupby("segment")
    t = g[variables].mean().round(2)
    t.insert(0, "clients", g.size())
    t.insert(1, "part (%)", (100 * g.size() / len(df)).round(1))
    t["(Income)"] = g["Income"].mean().round(0)
    t["(taux Response)"] = g["Response"].mean().round(3)
    return t.sort_values(variables[0], ascending=False)

df[VARIABLES_DEFAUT].describe().round(1).T[["mean", "std", "min", "max"]]

### 3. Le poids des variables dans la distance

**Question marketing :** *quand on dit que deux clients « se ressemblent », quelle variable parle ?*

**La notion.** La distance additionne les écarts au carré sur chaque variable. Or les dépenses se comptent en centaines, les enfants en unités : sur l'échelle brute, **une variable aux grands nombres écrase les autres**. **Standardiser**, c'est ramener chaque variable à la même échelle (moyenne 0, écart-type 1) : cela ne « nettoie » rien, cela **change le poids relatif des variables dans la distance**. Avant / après, ce ne sont pas les mêmes clients qui sont « proches ».

In [ ]:
a, b = 0, 1                                         # deux clients du tableau préparé
Z = pd.DataFrame(StandardScaler().fit_transform(df[VARIABLES_DEFAUT]), columns=VARIABLES_DEFAUT)

A = df.loc[a, VARIABLES_DEFAUT].astype(float)
B = df.loc[b, VARIABLES_DEFAUT].astype(float)
ecarts = pd.DataFrame({
    "client A": A,
    "client B": B,
    "écart² brut": ((A - B) ** 2).round(1),
    "écart² standardisé": ((Z.loc[a] - Z.loc[b]) ** 2).round(2),
})
ecarts["part brut (%)"] = (100 * ecarts["écart² brut"] / ecarts["écart² brut"].sum()).round(1)
ecarts["part standardisé (%)"] = (100 * ecarts["écart² standardisé"] / ecarts["écart² standardisé"].sum()).round(1)
display(ecarts)
print(f"Distance brute : {np.sqrt(ecarts['écart² brut'].sum()):.0f} | distance standardisée : {np.sqrt(ecarts['écart² standardisé'].sum()):.2f}")

**Question 1.** Sur l'échelle brute, quelle part de la distance entre ces deux clients vient des dépenses ? Et des enfants ? Après standardisation ? Lequel des deux calculs correspond à l'idée qu'un responsable marketing se fait de « se ressembler » ?

### 4. Le formulaire de segmentation

**Formulaire.** Cochez les variables, choisissez la standardisation et k, relancez la cellule. Réglages par défaut : les quatre variables de départ, standardisation activée, k = 4. **Remettez ces réglages avant la section 7.**

In [ ]:
#@title Réglages de la segmentation
Depenses_total = True  #@param {type:"boolean"}
NumDealsPurchases = True  #@param {type:"boolean"}
Enfants = True  #@param {type:"boolean"}
Age = True  #@param {type:"boolean"}
Part_web = False  #@param {type:"boolean"}
Income = False  #@param {type:"boolean"}
Recency = False  #@param {type:"boolean"}
Anciennete_mois = False  #@param {type:"boolean"}
NumWebVisitsMonth = False  #@param {type:"boolean"}
standardiser = True  #@param {type:"boolean"}
k = 4  #@param [2, 3, 4, 5, 6] {type:"raw"}

choix = {"Depenses_total": Depenses_total, "NumDealsPurchases": NumDealsPurchases, "Enfants": Enfants, "Age": Age,
         "Part_web": Part_web, "Income": Income, "Recency": Recency, "Anciennete_mois": Anciennete_mois,
         "NumWebVisitsMonth": NumWebVisitsMonth}
variables_segmentation = [v for v, coche in choix.items() if coche]
assert len(variables_segmentation) >= 2, "Cochez au moins deux variables."

segments, silhouette, inertie = segmenter(variables_segmentation, k, standardiser)
df["segment"] = segments
print("Variables :", ", ".join(variables_segmentation), "| standardisation :", "oui" if standardiser else "non", "| k =", k)
print(f"Silhouette : {silhouette:.3f}   (−1 à 1 ; au-dessous de 0,25 : coupures surtout arbitraires ; 0,25 à 0,50 : structure faible mais lisible)")
profil(segments, variables_segmentation)

Les numéros de segment sont arbitraires (0, 1, 2, 3 ne signifient rien) ; le tableau est trié par la première variable cochée. Les deux dernières colonnes, entre parenthèses, n'ont **pas** servi au calcul : elles serviront à la validation (section 8).

### 5. Avant / après standardisation

**Question marketing :** *si l'on oublie de standardiser, que deviennent les segments ?*

Même configuration, même k, avec et sans standardisation, côte à côte.

In [ ]:
seg_brut, sil_brut, _ = segmenter(VARIABLES_DEFAUT, 4, standardiser=False)
seg_std, sil_std, _ = segmenter(VARIABLES_DEFAUT, 4, standardiser=True)
print(f"Sans standardisation : silhouette {sil_brut:.3f}")
display(profil(seg_brut, VARIABLES_DEFAUT))
print(f"Avec standardisation : silhouette {sil_std:.3f}")
display(profil(seg_std, VARIABLES_DEFAUT))
print(f"Accord entre les deux découpages (ARI, 1 = identiques, 0 = sans rapport) : {adjusted_rand_score(seg_brut, seg_std):.2f}")

**Question 2.** Sans standardisation, que décrivent les quatre segments ? Les trois autres variables servent-elles à quelque chose ? La silhouette est pourtant bien plus élevée sans standardisation : est-ce que cela en fait une meilleure segmentation ?

## Bloc 2 — Interpréter et valider

### 6. Combien de groupes ?

**Question marketing :** *faut-il 2, 4 ou 6 segments ?*

**La notion.** La **silhouette** mesure si chaque client est plus proche de son groupe que du groupe voisin (de −1 à 1). La **stabilité à l'initialisation** vérifie que relancer l'algorithme avec d'autres points de départ (graines) donne le même découpage. Aucun des deux ne décide à votre place.

In [ ]:
lignes = []
for k_essai in [2, 3, 4, 5, 6]:
    seg_k, sil_k, _ = segmenter(VARIABLES_DEFAUT, k_essai)
    autres = [segmenter(VARIABLES_DEFAUT, k_essai, graine=g)[0] for g in range(10)]
    accords = [adjusted_rand_score(autres[i], autres[j]) for i in range(10) for j in range(i + 1, 10)]
    lignes.append({"k": k_essai, "silhouette": round(sil_k, 3),
                   "stabilité à l'initialisation (ARI moyen, 10 graines)": round(np.mean(accords), 3),
                   "plus petit segment (%)": round(100 * np.bincount(seg_k).min() / len(df), 1)})
choix_k = pd.DataFrame(lignes).set_index("k")
display(choix_k)

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(choix_k.index, choix_k["silhouette"], marker="o", color="#3b6ea5", linewidth=2)
ax.set_xticks(list(choix_k.index))
ax.set_ylim(0, 0.5)
ax.set_xlabel("nombre de segments k")
ax.set_ylabel("silhouette")
ax.set_title("La silhouette varie peu avec k, et reste faible", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

k = 6 a la meilleure silhouette, mais l'écart avec k = 4 est faible (0,316 contre 0,288) et toutes les valeurs restent basses. **k est une décision marketing** (combien de campagnes distinctes l'équipe peut-elle réellement mener ?), pas un verdict de la métrique.

**Question 3.** Que valent les silhouettes, et que signifie leur niveau ? Les découpages sont-ils stables à l'initialisation ? Si les indicateurs ne tranchent pas entre 4 et 6, qu'est-ce qui doit trancher ?

### 7. Les profils des segments

**Question marketing :** *à quoi ressemble chaque segment ?*

Réglages par défaut du formulaire (section 4). Le graphique montre, pour chaque segment, l'écart de chaque variable à la moyenne de tous les clients, en écarts-types : c'est la lecture la plus honnête d'un profil.

In [ ]:
segments, silhouette, _ = segmenter(variables_segmentation, k, standardiser)
df["segment"] = segments
tableau = profil(segments, variables_segmentation)
display(tableau)

Zs = pd.DataFrame(StandardScaler().fit_transform(df[variables_segmentation]), columns=variables_segmentation)
ecarts_types = Zs.assign(segment=segments).groupby("segment").mean().loc[tableau.index]

couleurs = ["#3b6ea5", "#c0392b", "#2e8b57", "#8e44ad", "#d68910", "#17a2b8", "#555555", "#b5651d", "#1f77b4"]
fig, ax = plt.subplots(figsize=(9, 3.8))
largeur = 0.8 / len(variables_segmentation)
positions = np.arange(len(ecarts_types))
for i, v in enumerate(variables_segmentation):
    ax.bar(positions + i * largeur, ecarts_types[v], width=largeur, color=couleurs[i], label=v)
ax.axhline(0, color="#7f7f7f", linewidth=1)
ax.set_xticks(positions + largeur * (len(variables_segmentation) - 1) / 2)
ax.set_xticklabels([f"segment {s}\n({tableau.loc[s, 'part (%)']} %)" for s in ecarts_types.index])
ax.set_ylabel("écart à la moyenne (en écarts-types)")
ax.set_title("Ce qui distingue chaque segment", loc="left")
ax.set_ylim(ecarts_types.values.min() * 1.15, ecarts_types.values.max() * 1.45)   # de la place pour la légende
ax.legend(frameon=False, ncol=2, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

**Question 4.** Pour chaque segment, citez la ou les variables qui s'écartent le plus de la moyenne, dans quel sens, et l'effectif. Y a-t-il un segment qui se distingue sur une seule variable ?

### 8. Les segments tiennent-ils sur ce qu'on n'a pas utilisé ?

**Question marketing :** *ces segments ont-ils un sens au-delà des quatre variables qui les ont fabriqués ?*

**La notion.** Un découpage est **utile** s'il sépare aussi les clients sur des variables qui n'ont pas servi à le construire : c'est la **validation externe**. Ici : le revenu, et le taux de réponse à la dernière campagne. Si les segments avaient été tirés au hasard, ces colonnes seraient identiques d'un segment à l'autre.

In [ ]:
validation = df.groupby("segment").agg(clients=("ID", "size"), revenu_moyen=("Income", "mean"),
                                      taux_reponse=("Response", "mean"), part_web=("Part_web", "mean"),
                                      campagnes_acceptees=("Campagnes_acceptees", "mean")).round(3).loc[tableau.index]
validation["revenu_moyen"] = validation["revenu_moyen"].round(0)
print(f"Pour mémoire, tous clients confondus : revenu moyen {df['Income'].mean():.0f}, taux de réponse {df['Response'].mean():.3f}")
validation

**Question 5.** Les segments diffèrent-ils sur le revenu et le taux de réponse, alors que ces variables n'ont pas servi ? Lequel est le plus réactif aux campagnes, lequel l'est le moins ? Qu'est-ce que cela apporte à la question « à qui envoyer la prochaine campagne ? », et qu'est-ce que cela ne prouve pas ?

### 9. Nommer les segments

**Question marketing :** *comment en parler à l'équipe ?*

**La notion.** Un nom de segment est une **interprétation** : il doit pouvoir être justifié par deux ou trois chiffres du profil, et il ne doit rien affirmer que le profil ne montre pas (« fidèles », « engagés », « à fort potentiel » sont des promesses, pas des observations).

**Question 6.** Donnez à chaque segment un nom en trois mots au plus, les deux chiffres qui le justifient, et une action marketing. Puis dites quel segment vous laisseriez de côté pour la prochaine campagne, et pourquoi.

### 10. Changer les variables, changer les segments

**Démonstration en séance : une variable de bruit.** On ajoute `Recency` aux quatre variables par défaut (même k, même standardisation). Le notebook 01 avait montré une variable plate, sans lien avec les autres.

In [ ]:
seg_rec, sil_rec, _ = segmenter(VARIABLES_DEFAUT + ["Recency"], 4)
print(f"Avec Recency : silhouette {sil_rec:.3f} (sans : {sil_std:.3f}) | accord avec la segmentation par défaut (ARI) : {adjusted_rand_score(seg_std, seg_rec):.2f}")
profil(seg_rec, VARIABLES_DEFAUT + ["Recency"])

**Question 7.** Que fait `Recency` aux segments ? Repérez deux segments qui ont presque les mêmes dépenses : qu'est-ce qui les sépare ? Que conclure sur la phrase « les segments sont une construction » ?

**Cellule optionnelle 1 — variante C** (hors minutage de la séance) : `Part_web` remplace `Age`. Un axe « canal » remplace l'axe « âge ».

In [ ]:
# Optionnel : variante C
VARIABLES_C = ["Depenses_total", "NumDealsPurchases", "Enfants", "Part_web"]
seg_c, sil_c, _ = segmenter(VARIABLES_C, 4)
print(f"Variante C : silhouette {sil_c:.3f} | accord avec la segmentation par défaut (ARI) : {adjusted_rand_score(seg_std, seg_c):.2f}")
profil(seg_c, VARIABLES_C)

**Cellule optionnelle 2 — un doublon** (hors minutage) : `Income` s'ajoute aux quatre variables par défaut, alors qu'il redit largement `Depenses_total` (corrélation 0,79).

In [ ]:
# Optionnel : ajout d'Income
seg_inc, sil_inc, _ = segmenter(VARIABLES_DEFAUT + ["Income"], 4)
print(f"Avec Income : silhouette {sil_inc:.3f} | accord avec la segmentation par défaut (ARI) : {adjusted_rand_score(seg_std, seg_inc):.2f}")
profil(seg_inc, VARIABLES_DEFAUT + ["Income"])

### 11. Synthèse métier

- Quatre segments **stables à l'initialisation et utiles** (silhouette 0,29, accord entre graines 0,99) : gros acheteurs sans enfant (26 %, 30 % de réponse), familles en promotion (14 %), seniors petit budget (26 %, 5 % de réponse), jeunes foyers petit budget (34 %).
- Ils tiennent sur ce qu'ils n'ont pas vu : revenu de 37 600 à 75 100, réponse de 5 % à 30 %. C'est l'argument de leur utilité.
- **Standardiser change le poids des variables dans la distance** : sans elle, les dépenses écrasent tout et les segments sont des tranches de chiffre d'affaires, avec une silhouette flatteuse mais vide.
- Les silhouettes sont **faibles** : il n'y a pas de groupes naturels dans ces données. k = 6 a la meilleure silhouette, de peu ; **k est une décision marketing** (combien de campagnes distinctes sont réalistes ?), pas un verdict de la métrique.
- La variante C (`Part_web` à la place de `Age`) obtient la **même silhouette** que la configuration par défaut (0,287 contre 0,288) mais seulement **0,41 d'accord** avec elle : deux choix de variables raisonnables donnent des segmentations différentes que la métrique ne départage pas. **Le choix des variables est une décision métier.**
- Un nom de segment se justifie par deux chiffres et ne promet rien.

### 12. Limites

- Les segments dépendent des quatre variables choisies ; une autre configuration (C) donne une autre lecture, aussi valable.
- Stables à l'initialisation (accord 0,99), les segments le sont moins **au rééchantillonnage des clients** : en refaisant la segmentation sur 20 tirages avec remise de la base, l'accord avec la segmentation de référence vaut 0,82 en moyenne (de 0,49 à 0,96). Les grands traits (gros acheteurs, promotions, âge) tiennent ; les frontières entre segments bougent avec l'échantillon, et un client proche d'une frontière peut changer de segment.
- `Depenses_total` et `NumDealsPurchases` portent sur une fenêtre de deux ans non datée ; un client récent a mécaniquement moins dépensé (notebook 01).
- Les taux de réponse par segment décrivent la campagne passée, envoyée au hasard ; ils ne prédisent pas l'effet d'une campagne ciblée.
- Les données sont simulées ; certaines régularités (profils dupliqués, récence uniforme) viennent de la fabrication du fichier.

### En autonomie (30 minutes) — brique projet 4 : la fiche segmentation du groupe

Chaque groupe rend une **fiche segmentation** d'une page, qui entrera dans le projet final :

1. **Segments retenus** : configuration de variables (par défaut ou variante), standardisation, et le tableau des profils (effectif, part, moyennes, revenu et taux de réponse).
2. **k choisi et justifié** : la silhouette et la stabilité pour les k essayés, puis l'argument marketing qui tranche (combien de campagnes distinctes le groupe juge réalistes).
3. **Nommage argumenté** : pour chaque segment, un nom de trois mots au plus et les deux chiffres qui le justifient ; aucun mot qui promet (« fidèles », « à potentiel »).
4. **Une action marketing par segment**, et le segment qu'on laisse de côté pour la prochaine campagne, avec la raison.
5. **Limites** : trois phrases au moins (construction dépendante des variables, silhouettes faibles, fenêtre de deux ans non datée, taux de réponse décrivant une campagne passée).

**Exercices facultatifs** (hors fiche) : (a) fiche persona du segment « acheteurs web » de la variante C (cellule optionnelle 1) : trois chiffres, un nom, une action, ce que la fiche ne peut pas affirmer ; (b) lecture critique : réglages par défaut, standardisation désactivée, deux phrases à dire à un collègue qui présenterait ces segments-là.